# Stage 1 Labeling V2

这个 notebook 服务 `label_v2_relaxed_text_observed`：

- 保持 `text-observed`
- 允许 `partial retreat`
- 允许 `anchor-self retreat`
- 以 `single-pass LLM` 为主
- 仅边界样本进入 second pass
- 所有新输出统一写入 `labeling_new/`


当前正式执行说明：
- 全样本 `primary-only` 版本
- 已修复 second-term `event_time_utc` mixed parsing 问题
- 不包含 second pass
- finalize 前包含 second-term manual scope exclusion 步骤

In [ ]:
from pathlib import Path
import json
import os
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

LABELING_DIR = PROJECT_ROOT / "labeling_new"
LABELING_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_PATH = PROJECT_ROOT / "data" / "processed" / "standardized" / "03_candidate_events_standardized.csv"
AUDIT_LIST_PATH = PROJECT_ROOT / "data" / "manual" / "key_event_audit_list_v1.csv"
PROMPTS_PATH = LABELING_DIR / "labeling_v2_prompts.md"

RULE_OUTPUT_PATH = LABELING_DIR / "03_candidate_events_rule_pretag_v2.csv"
LABEL_INPUT_PATH = LABELING_DIR / "03_candidate_events_labeling_input_v2.csv"
PRIMARY_BATCH_JSONL_PATH = LABELING_DIR / "llm_labeling_batch_input_primary_v2.jsonl"
PRIMARY_RESPONSE_PATH = LABELING_DIR / "llm_labeling_response_primary_v2.csv"
SECOND_PASS_QUEUE_PATH = LABELING_DIR / "label_second_pass_queue_v2.csv"
SECOND_PASS_BATCH_JSONL_PATH = LABELING_DIR / "llm_labeling_batch_input_second_pass_v2.jsonl"
SECOND_PASS_RESPONSE_PATH = LABELING_DIR / "llm_labeling_response_second_pass_v2.csv"
FINAL_LABEL_PATH = LABELING_DIR / "labels_relaxed_text_observed_v2.csv"
QC_AUDIT_PATH = LABELING_DIR / "label_qc_audit_v2.csv"
AUDIT_MATCH_PATH = LABELING_DIR / "key_event_audit_matches_v2.csv"
AUDIT_COVERAGE_PATH = LABELING_DIR / "key_event_audit_coverage_v2.csv"

TERM_FILTER = None

DEEPSEEK_MODEL = "deepseek-v4-flash"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
SAVE_EVERY = 10
SLEEP_SECONDS = 0.8
MAX_RETRIES = 2
PILOT_SIZE = 20
PRIMARY_BATCH_SIZE = 60
PRIMARY_BATCH_NUMBER = 1
SECOND_PASS_BATCH_SIZE = 40
SECOND_PASS_BATCH_NUMBER = 1
MANUAL_SCOPE_EXCLUSION_IDS = [
    "SECOND_TERM_TRUTHSOCIAL_114675841423085927",
    "SECOND_TERM_TRUTHSOCIAL_114730186433008075",
    "SECOND_TERM_TRUTHSOCIAL_114739003459183372",
    "SECOND_TERM_TRUTHSOCIAL_115362196088273474",
]

def apply_term_filter(frame: pd.DataFrame) -> pd.DataFrame:
    if TERM_FILTER is None:
        return frame.copy()
    if "term_id" not in frame.columns:
        return frame.copy()
    return frame.loc[frame["term_id"].astype(str).eq(TERM_FILTER)].copy()

print("Project root:", PROJECT_ROOT)
print("Labeling v2 dir:", LABELING_DIR)
print("Term filter:", TERM_FILTER)

## V2 Rules

本版固定规则：
- `partial retreat` 算正类
- `anchor-self retreat` 算正类
- 只有 `deal / progress / productive` 不算正类
- mixed rhetoric 不默认判 `0`
- 判断依据必须是明确的 `de-escalation action`

In [ ]:
RETREAT_ACTION_PATTERNS = {
    r"\bdelay(?:ed|ing|s)?\b": 3,
    r"\bpaus(?:e|ed|ing)\b": 3,
    r"\bsuspend(?:ed|ing|s)?\b": 3,
    r"\bexempt(?:ion|ions)?\b": 3,
    r"\bwaiver(?:s)?\b": 3,
    r"\btariff[- ]free\b": 4,
    r"\bnot be charged\b": 4,
    r"\bremain at\b": 2,
    r"\bwill not be charged\b": 4,
}

PRESSURE_THEME_PATTERNS = {
    r"\btariff(?:s)?\b": 1,
    r"\btrade\b": 1,
    r"\bchina\b": 1,
    r"\bmexico\b": 1,
    r"\bcanada\b": 1,
    r"\beu\b": 1,
    r"\beuropean union\b": 1,
    r"\bimport(?:s)?\b": 1,
    r"\bexport(?:s)?\b": 1,
    r"\bsanction(?:s)?\b": 1,
    r"\baluminum\b": 1,
    r"\bsteel\b": 1,
}


def normalize_text(value: str) -> str:
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_section(source_text: str, header: str) -> str:
    if pd.isna(source_text):
        return ""
    text = str(source_text)
    marker = f"[{header}]"
    if marker not in text:
        return ""
    chunk = text.split(marker, 1)[1]
    next_headers = ["[Prior same-theme context, past 7d]", "[Anchor]", "[Follow-up within 48h]"]
    end_at = len(chunk)
    for item in next_headers:
        if item == marker:
            continue
        idx = chunk.find(item)
        if idx != -1:
            end_at = min(end_at, idx)
    return chunk[:end_at].strip()


def pattern_hits(text: str, patterns: dict[str, int]) -> list[str]:
    return [pattern for pattern in patterns if re.search(pattern, text)]


candidate_df = apply_term_filter(pd.read_csv(CANDIDATE_PATH))
candidate_df["source_text"] = candidate_df["source_text"].fillna("")

anchor_text = candidate_df["source_text"].map(lambda x: extract_section(x, "Anchor"))
follow_up_text = candidate_df["source_text"].map(lambda x: extract_section(x, "Follow-up within 48h"))
combined_text = (anchor_text.fillna("") + " " + follow_up_text.fillna("")).map(normalize_text)

candidate_df["rule_retreat_hits_v2"] = combined_text.map(lambda x: "|".join(pattern_hits(x, RETREAT_ACTION_PATTERNS)))
candidate_df["rule_theme_hits_v2"] = combined_text.map(lambda x: "|".join(pattern_hits(x, PRESSURE_THEME_PATTERNS)))
candidate_df["rule_retreat_score_v2"] = combined_text.map(lambda x: sum(RETREAT_ACTION_PATTERNS[p] for p in pattern_hits(x, RETREAT_ACTION_PATTERNS)))
candidate_df["rule_theme_score_v2"] = combined_text.map(lambda x: sum(PRESSURE_THEME_PATTERNS[p] for p in pattern_hits(x, PRESSURE_THEME_PATTERNS)))
candidate_df["rule_label_v2"] = (
    (candidate_df["rule_retreat_score_v2"] >= 3) & (candidate_df["rule_theme_score_v2"] >= 1)
).astype(int)

candidate_df.to_csv(RULE_OUTPUT_PATH, index=False)
display(candidate_df[["event_id", "rule_label_v2", "rule_retreat_hits_v2", "rule_theme_hits_v2"]].head())
print("Wrote:", RULE_OUTPUT_PATH)

## Prior Context

这里为每条候选事件构造 `past 7d same-theme context`，用于帮助判断 `anchor-self retreat`。

In [ ]:
rule_df = apply_term_filter(pd.read_csv(RULE_OUTPUT_PATH))
rule_df["event_time_utc"] = pd.to_datetime(rule_df["event_time_utc"], errors="coerce", utc=True, format="mixed")


def build_prior_context(frame: pd.DataFrame, idx: int, lookback_days: int = 7, max_items: int = 5) -> str:
    row = frame.iloc[idx]
    ts = row["event_time_utc"]
    if pd.isna(ts):
        return ""

    start_ts = ts - pd.Timedelta(days=lookback_days)
    same_term = frame["term_id"] == row["term_id"]
    earlier = frame["event_time_utc"] < ts
    within_window = frame["event_time_utc"] >= start_ts
    same_theme = frame["rule_theme_score_v2"] > 0

    candidates = frame.loc[same_term & earlier & within_window & same_theme, ["event_time_utc", "source_text"]].tail(max_items)
    lines = []
    for _, prior_row in candidates.iterrows():
        anchor = extract_section(prior_row["source_text"], "Anchor") or str(prior_row["source_text"])[:280]
        lines.append(f"{prior_row['event_time_utc']} | {anchor}")
    return "\n".join(lines)


rule_df["prior_same_theme_context_7d"] = [build_prior_context(rule_df, idx) for idx in range(len(rule_df))]
rule_df.to_csv(LABEL_INPUT_PATH, index=False)
display(rule_df[["event_id", "prior_same_theme_context_7d"]].head())
print("Wrote:", LABEL_INPUT_PATH)

## Prompt Template

本 notebook 的 prompt 从 `labeling_new/labeling_v2_prompts.md` 读取。
这样你修改 prompt 文件后，可以直接在 notebook 中重新加载。

In [ ]:
def extract_prompt_block(markdown_text: str, heading: str, required: bool = True) -> str:
    pattern = rf"## {re.escape(heading)}\n\n```text\n(.*?)\n```"
    match = re.search(pattern, markdown_text, flags=re.S)
    if not match:
        if required:
            raise ValueError(f"Missing prompt section: {heading}")
        return ""
    return match.group(1).strip()


prompt_markdown = PROMPTS_PATH.read_text(encoding="utf-8")
PRIMARY_SYSTEM_PROMPT = extract_prompt_block(prompt_markdown, "Primary System Prompt")
SECOND_PASS_SYSTEM_PROMPT = extract_prompt_block(prompt_markdown, "Second Pass System Prompt", required=False)

print("Loaded prompts from:", PROMPTS_PATH)
print("Primary prompt preview:")
print(PRIMARY_SYSTEM_PROMPT[:600] + "...")


def build_primary_user_prompt(row: pd.Series) -> str:
    return f"""json only

Event metadata:
- event_id: {row['event_id']}
- source: {row.get('source', '')}
- term_id: {row.get('term_id', '')}
- event_time_utc: {row.get('event_time_utc', '')}
- rule_label_v2: {row.get('rule_label_v2', '')}

Task:
Decide whether this event shows a retreat under the relaxed v2 definition.

[Prior same-theme context, past 7d]
{row.get('prior_same_theme_context_7d', '')}

[Source text]
{row.get('source_text', '')}
"""

## DeepSeek Config

本版沿用旧的接入方式：
- `OpenAI SDK`
- `base_url = https://api.deepseek.com/v1`
- `model = deepseek-v4-flash`
- `response_format = {"type": "json_object"}`

In [ ]:
api_key = os.environ.get("DEEPSEEK_API_KEY")
if api_key:
    print("DEEPSEEK_API_KEY is set. Ready to run pilot batch.")
else:
    print("DEEPSEEK_API_KEY is missing. In terminal run:")
    print('export DEEPSEEK_API_KEY="your_key_here"')

## Primary Pass Prep

这里准备 primary pass 的 response template 和 batch jsonl。

In [ ]:
label_input_df = apply_term_filter(pd.read_csv(LABEL_INPUT_PATH))

primary_template_cols = [
    "event_id",
    "out_of_scope",
    "label_retreat",
    "retreat_type",
    "context_basis",
    "confidence",
    "evidence_span",
    "prior_pressure_span",
    "review_flag",
    "review_reason",
    "model_name",
    "raw_response",
]

if PRIMARY_RESPONSE_PATH.exists():
    existing_primary_template_df = pd.read_csv(PRIMARY_RESPONSE_PATH)
else:
    existing_primary_template_df = pd.DataFrame(columns=primary_template_cols)

if existing_primary_template_df.empty:
    primary_template_df = label_input_df[["event_id"]].copy()
    for col in primary_template_cols[1:]:
        primary_template_df[col] = pd.NA
else:
    primary_template_df = (
        label_input_df[["event_id"]]
        .merge(existing_primary_template_df, on="event_id", how="left")
    )
    for col in primary_template_cols:
        if col not in primary_template_df.columns:
            primary_template_df[col] = pd.NA
    primary_template_df = primary_template_df[primary_template_cols]

primary_template_df.to_csv(PRIMARY_RESPONSE_PATH, index=False)

with PRIMARY_BATCH_JSONL_PATH.open("w", encoding="utf-8") as f:
    for _, row in label_input_df.iterrows():
        payload = {
            "event_id": row["event_id"],
            "system_prompt": PRIMARY_SYSTEM_PROMPT,
            "user_prompt": build_primary_user_prompt(row),
        }
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

display(primary_template_df.head())
print("Wrote:", PRIMARY_RESPONSE_PATH)
print("Wrote:", PRIMARY_BATCH_JSONL_PATH)

## Primary LLM Runtime

这一段是可直接跑的 DeepSeek 调用逻辑，支持断点续跑。

In [ ]:
from typing import Any
from openai import OpenAI


TEXT_RESULT_COLUMNS = [
    "event_id",
    "retreat_type",
    "context_basis",
    "evidence_span",
    "prior_pressure_span",
    "review_flag",
    "review_reason",
    "model_name",
    "raw_response",
]


def load_frames(label_input_path: Path, response_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    label_input = apply_term_filter(pd.read_csv(label_input_path))
    template = pd.read_csv(response_path)
    template = label_input[["event_id"]].merge(template, on="event_id", how="left")
    for col in TEXT_RESULT_COLUMNS:
        if col in template.columns:
            template[col] = template[col].astype("string")
    for col in ["out_of_scope", "label_retreat"]:
        if col in template.columns:
            template[col] = pd.to_numeric(template[col], errors="coerce")
    if "confidence" in template.columns:
        template["confidence"] = pd.to_numeric(template["confidence"], errors="coerce")
    return label_input, template


def build_deepseek_client() -> OpenAI:
    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("DEEPSEEK_API_KEY is not set.")
    return OpenAI(api_key=api_key, base_url=DEEPSEEK_BASE_URL)


def strip_code_fences(text: str) -> str:
    if not text:
        return text
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def parse_llm_json(raw_content: str, event_id: str) -> dict[str, Any]:
    content = strip_code_fences(raw_content)
    if not content:
        raise ValueError("Empty content returned by model.")

    parsed = json.loads(content)

    required = [
        "out_of_scope",
        "label_retreat",
        "retreat_type",
        "context_basis",
        "confidence",
        "evidence_span",
        "prior_pressure_span",
        "review_flag",
        "review_reason",
    ]
    for field in required:
        if field not in parsed:
            raise ValueError(f"Missing field: {field}")

    out_of_scope = int(parsed["out_of_scope"])
    label_retreat = int(parsed["label_retreat"])
    if out_of_scope not in (0, 1):
        raise ValueError(f"Invalid out_of_scope: {out_of_scope}")
    if label_retreat not in (0, 1):
        raise ValueError(f"Invalid label_retreat: {label_retreat}")

    retreat_type = str(parsed["retreat_type"]).strip()
    if retreat_type not in {"full", "partial", "anchor_self", "none"}:
        raise ValueError(f"Invalid retreat_type: {retreat_type}")

    context_basis = str(parsed["context_basis"]).strip()
    if context_basis not in {"follow_up", "anchor", "both", "none"}:
        raise ValueError(f"Invalid context_basis: {context_basis}")

    confidence = max(0.0, min(1.0, float(parsed["confidence"])))
    evidence_span = str(parsed.get("evidence_span", "") or "")
    prior_pressure_span = str(parsed.get("prior_pressure_span", "") or "")
    review_flag = str(parsed.get("review_flag", "yes") or "yes").strip().lower()
    if review_flag not in {"yes", "no"}:
        review_flag = "yes"
    review_reason = str(parsed.get("review_reason", "") or "")

    # Mutual exclusivity rules to prevent logically inconsistent rows from
    # silently entering the label table.
    if out_of_scope == 1 and label_retreat != 0:
        raise ValueError("out_of_scope=1 must imply label_retreat=0.")
    if out_of_scope == 1 and retreat_type != "none":
        raise ValueError("out_of_scope=1 must imply retreat_type='none'.")
    if out_of_scope == 1 and context_basis != "none":
        raise ValueError("out_of_scope=1 must imply context_basis='none'.")
    if label_retreat == 0 and retreat_type != "none":
        raise ValueError("label_retreat=0 must imply retreat_type='none'.")
    if label_retreat == 0 and context_basis != "none":
        raise ValueError("label_retreat=0 must imply context_basis='none'.")

    if label_retreat == 1 and not evidence_span.strip():
        raise ValueError("Positive label missing evidence_span.")
    if retreat_type == "anchor_self" and not prior_pressure_span.strip():
        raise ValueError("anchor_self missing prior_pressure_span.")
    if label_retreat == 1 and out_of_scope == 1:
        raise ValueError("Positive label cannot also be out_of_scope.")

    return {
        "event_id": event_id,
        "out_of_scope": out_of_scope,
        "label_retreat": label_retreat,
        "retreat_type": retreat_type,
        "context_basis": context_basis,
        "confidence": confidence,
        "evidence_span": evidence_span,
        "prior_pressure_span": prior_pressure_span,
        "review_flag": review_flag,
        "review_reason": review_reason,
        "raw_response": content,
    }


def call_deepseek_label(client: OpenAI, row: pd.Series, system_prompt: str, user_prompt_builder, max_retries: int = MAX_RETRIES) -> dict[str, Any]:
    user_prompt = user_prompt_builder(row)
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=DEEPSEEK_MODEL,
                temperature=0,
                max_tokens=500,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw_content = response.choices[0].message.content
            parsed = parse_llm_json(raw_content, row["event_id"])
            parsed["model_name"] = DEEPSEEK_MODEL
            return parsed
        except Exception:
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
            else:
                raise


def get_unlabeled_rows(label_input_df: pd.DataFrame, template_df: pd.DataFrame) -> pd.DataFrame:
    pending_ids = set(template_df.loc[template_df["label_retreat"].isna(), "event_id"])
    return label_input_df.loc[label_input_df["event_id"].isin(pending_ids)].copy()


def save_template(template_df: pd.DataFrame, response_path: Path) -> None:
    template_df.to_csv(response_path, index=False)


def write_result_to_template(template_df: pd.DataFrame, event_id: str, result: dict[str, Any]) -> None:
    idx = template_df.index[template_df["event_id"] == event_id][0]
    for key, value in result.items():
        template_df.at[idx, key] = value


def run_labeling_batch(
    rows_to_run: pd.DataFrame,
    response_path: Path,
    system_prompt: str,
    user_prompt_builder,
    sleep_seconds: float = SLEEP_SECONDS,
    save_every: int = SAVE_EVERY,
) -> pd.DataFrame:
    if rows_to_run.empty:
        print("No rows to label in this batch.")
        return pd.read_csv(response_path)

    client = build_deepseek_client()
    _, template_df = load_frames(LABEL_INPUT_PATH if response_path == PRIMARY_RESPONSE_PATH else SECOND_PASS_QUEUE_PATH, response_path)

    completed = 0
    failed_event_ids = []
    for _, row in rows_to_run.iterrows():
        event_id = row["event_id"]
        try:
            result = call_deepseek_label(client, row, system_prompt=system_prompt, user_prompt_builder=user_prompt_builder)
            write_result_to_template(template_df, event_id, result)
            completed += 1
        except Exception as exc:
            failed_event_ids.append(event_id)
            print(f"[FAILED] {event_id}: {exc}")

        if completed > 0 and completed % save_every == 0:
            save_template(template_df, response_path)
            print(f"Saved progress after {completed} completed rows.")

        time.sleep(sleep_seconds)

    save_template(template_df, response_path)
    print(f"Batch finished. completed={completed}, failed={len(failed_event_ids)}")
    if failed_event_ids:
        print("Failed event_ids:", failed_event_ids[:20])
    return template_df

## Primary Pilot Batch

先跑 `10-20` 条，不要一上来就全量。

In [ ]:
label_input_df, primary_template_df = load_frames(LABEL_INPUT_PATH, PRIMARY_RESPONSE_PATH)
pending_primary_df = get_unlabeled_rows(label_input_df, primary_template_df)

pilot_primary_df = (
    pending_primary_df
    .sort_values(["rule_label_v2", "event_time_utc"], ascending=[False, True])
    .head(PILOT_SIZE)
    .copy()
)

print(f"pending primary rows: {len(pending_primary_df):,}")
print(f"pilot primary rows: {len(pilot_primary_df):,}")
display(pilot_primary_df[["event_id", "rule_label_v2", "rule_retreat_hits_v2", "rule_theme_hits_v2"]].head(20))

## Primary Multi-Round Batch

这里按“未完成样本 + 当前轮编号”切出一轮 primary batch。

你只需要改：
- `PRIMARY_BATCH_SIZE`
- `PRIMARY_BATCH_NUMBER`

然后重复执行这一段和下一段运行单元即可。

In [ ]:
label_input_df, primary_template_df = load_frames(LABEL_INPUT_PATH, PRIMARY_RESPONSE_PATH)
pending_primary_df = get_unlabeled_rows(label_input_df, primary_template_df)
pending_primary_df = pending_primary_df.sort_values(["rule_label_v2", "event_time_utc"], ascending=[False, True]).reset_index(drop=True)

total_pending = len(pending_primary_df)
estimated_total_rounds = int(np.ceil(total_pending / PRIMARY_BATCH_SIZE)) if PRIMARY_BATCH_SIZE > 0 else 0
start_idx = max((PRIMARY_BATCH_NUMBER - 1) * PRIMARY_BATCH_SIZE, 0)
end_idx = start_idx + PRIMARY_BATCH_SIZE
current_primary_batch_df = pending_primary_df.iloc[start_idx:end_idx].copy()

estimated_minutes = len(current_primary_batch_df) * (SLEEP_SECONDS + 2.5) / 60 if len(current_primary_batch_df) else 0.0

print(f"pending primary rows: {total_pending:,}")
print(f"PRIMARY_BATCH_SIZE = {PRIMARY_BATCH_SIZE}")
print(f"PRIMARY_BATCH_NUMBER = {PRIMARY_BATCH_NUMBER}")
print(f"estimated total rounds: {estimated_total_rounds}")
print(f"current slice: [{start_idx}:{end_idx})")
print(f"current primary batch rows: {len(current_primary_batch_df):,}")
print(f"rough runtime estimate: {estimated_minutes:.1f} minutes")

display(current_primary_batch_df[[
    "event_id", "rule_label_v2", "rule_retreat_hits_v2", "rule_theme_hits_v2"
]].head(20))

## Next Round Suggestion

这个辅助单元不会运行 LLM，只会根据当前进度告诉你下一轮建议编号。

In [ ]:
label_input_df, primary_template_df = load_frames(LABEL_INPUT_PATH, PRIMARY_RESPONSE_PATH)
pending_primary_df = get_unlabeled_rows(label_input_df, primary_template_df)
total_primary = len(label_input_df)
done_primary = int(primary_template_df["label_retreat"].notna().sum()) if "label_retreat" in primary_template_df.columns else 0
next_primary_batch_number = 1 if len(pending_primary_df) > 0 else None

if SECOND_PASS_QUEUE_PATH.exists():
    second_pass_full_df = pd.read_csv(SECOND_PASS_QUEUE_PATH)
else:
    second_pass_full_df = pd.DataFrame()

if SECOND_PASS_RESPONSE_PATH.exists():
    second_pass_template_df = pd.read_csv(SECOND_PASS_RESPONSE_PATH)
else:
    second_pass_template_df = pd.DataFrame(columns=["event_id", "label_retreat"])

if not second_pass_full_df.empty:
    second_pass_template_df = second_pass_full_df[["event_id"]].merge(second_pass_template_df, on="event_id", how="left")

if not second_pass_full_df.empty and "label_retreat" in second_pass_template_df.columns:
    done_second_ids = set(second_pass_template_df.loc[second_pass_template_df["label_retreat"].notna(), "event_id"])
    pending_second_df = second_pass_full_df.loc[~second_pass_full_df["event_id"].isin(done_second_ids)].copy()
    done_second = int(second_pass_template_df["label_retreat"].notna().sum())
else:
    pending_second_df = second_pass_full_df.copy()
    done_second = 0

next_second_batch_number = 1 if len(pending_second_df) > 0 else None
remaining_primary_rounds = int(np.ceil(len(pending_primary_df) / PRIMARY_BATCH_SIZE)) if PRIMARY_BATCH_SIZE > 0 else 0
remaining_second_rounds = int(np.ceil(len(pending_second_df) / SECOND_PASS_BATCH_SIZE)) if SECOND_PASS_BATCH_SIZE > 0 else 0

print("Primary progress")
print(f"- total rows: {total_primary:,}")
print(f"- completed rows: {done_primary:,}")
print(f"- pending rows: {len(pending_primary_df):,}")
print(f"- suggested next PRIMARY_BATCH_NUMBER: {next_primary_batch_number}")
print(f"- estimated remaining rounds at current batch size: {remaining_primary_rounds}")
print("- note: after each completed run, rerun the batch selection cell and keep PRIMARY_BATCH_NUMBER = 1 unless you intentionally want to skip within the current pending snapshot")

print("\nSecond-pass progress")
print(f"- queue rows: {len(second_pass_full_df):,}")
print(f"- completed rows: {done_second:,}")
print(f"- pending rows: {len(pending_second_df):,}")
print(f"- suggested next SECOND_PASS_BATCH_NUMBER: {next_second_batch_number}")
print(f"- estimated remaining rounds at current batch size: {remaining_second_rounds}")
print("- note: after each completed run, rerun the second-pass batch selection cell and keep SECOND_PASS_BATCH_NUMBER = 1 unless you intentionally want to skip within the current pending snapshot")

In [ ]:
primary_result_df = run_labeling_batch(
    current_primary_batch_df,
    response_path=PRIMARY_RESPONSE_PATH,
    system_prompt=PRIMARY_SYSTEM_PROMPT,
    user_prompt_builder=build_primary_user_prompt,
    sleep_seconds=SLEEP_SECONDS,
    save_every=5,
)

display(
    primary_result_df.loc[primary_result_df["event_id"].isin(current_primary_batch_df["event_id"]), [
        "event_id", "out_of_scope", "label_retreat", "retreat_type", "context_basis",
        "confidence", "evidence_span", "prior_pressure_span", "review_flag", "review_reason"
    ]].head(20)
)

## Manual Scope Exclusions

这里对当前已确认不属于 `economic-policy de-escalation` 的 second-term scope drift 正类做手动剔除。

本步骤会直接回写 primary response csv，然后再进入 finalize。

In [ ]:
if not PRIMARY_RESPONSE_PATH.exists():
    raise FileNotFoundError(f"Primary response file not found: {PRIMARY_RESPONSE_PATH}")

primary_df = pd.read_csv(PRIMARY_RESPONSE_PATH)

mask = primary_df["event_id"].isin(MANUAL_SCOPE_EXCLUSION_IDS)
matched = int(mask.sum())

if matched:
    primary_df.loc[mask, "out_of_scope"] = 1
    primary_df.loc[mask, "label_retreat"] = 0
    primary_df.loc[mask, "retreat_type"] = "none"
    primary_df.loc[mask, "context_basis"] = "none"
    primary_df.loc[mask, "confidence"] = 0.95
    primary_df.loc[mask, "evidence_span"] = ""
    primary_df.loc[mask, "prior_pressure_span"] = ""
    primary_df.loc[mask, "review_flag"] = "yes"
    primary_df.loc[mask, "review_reason"] = "manual_scope_exclusion"
    primary_df.to_csv(PRIMARY_RESPONSE_PATH, index=False)

print("Manual scope exclusions configured:", len(MANUAL_SCOPE_EXCLUSION_IDS))
print("Matched rows updated:", matched)
display(
    primary_df.loc[primary_df["event_id"].isin(MANUAL_SCOPE_EXCLUSION_IDS), [
        "event_id", "out_of_scope", "label_retreat", "retreat_type",
        "context_basis", "confidence", "review_flag", "review_reason"
    ]]
)
print("Updated:", PRIMARY_RESPONSE_PATH)

## Finalize

这里直接使用 primary 结果输出最终 `v2` 标签表与 QC 表。

本版本不使用 second pass。

In [ ]:
label_input_df = apply_term_filter(pd.read_csv(LABEL_INPUT_PATH))
primary_df = pd.read_csv(PRIMARY_RESPONSE_PATH) if PRIMARY_RESPONSE_PATH.exists() else pd.DataFrame(columns=["event_id"])

final_df = label_input_df.merge(primary_df, on="event_id", how="left", suffixes=("", "_primary"))

qc_rows = []
for _, row in final_df.iterrows():
    qc_rows.append(
        {
            "event_id": row["event_id"],
            "label_retreat": row.get("label_retreat"),
            "confidence": row.get("confidence"),
            "has_evidence_span": int(bool(str(row.get("evidence_span", "")).strip())),
            "anchor_self_missing_prior_pressure": int(
                str(row.get("retreat_type", "")).strip() == "anchor_self"
                and not str(row.get("prior_pressure_span", "")).strip()
            ),
            "needs_manual_spot_check": int(float(row.get("confidence", 0.0) or 0.0) < 0.75),
        }
    )

qc_df = pd.DataFrame(qc_rows)
final_df.to_csv(FINAL_LABEL_PATH, index=False)
qc_df.to_csv(QC_AUDIT_PATH, index=False)

display(final_df.head())
display(qc_df.head())
print("Wrote:", FINAL_LABEL_PATH)
print("Wrote:", QC_AUDIT_PATH)